# Graph diagnostics: homophily, Moran’s I, and neighbor-mean baseline

> This notebook quantifies graph signal strength for the adjusted phenotype and evaluates a simple neighbor-mean baseline against an MLP. If homophily or Moran’s I are low, a GCN is unlikely to help unless it can learn to down-weight neighbors (e.g., attention/teleport).

In [5]:
# Setup and data loading
import json, numpy as np, pandas as pd
import scipy.sparse as sp
from typing import Dict, Any

import torch

from src.data import load_data
from src.graph import build_adjacency
from src.utils import to_sparse, _pearson_corr

# Config (adjust as needed)
CONFIG_PATH = 'config_nested.json'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

base = CFG['base_train']
paths = base['paths']
target_col = base.get('target_column', 'y_adjusted')
eval_col = base.get('eval_target_column', 'y_mean')

# Load data (includes GRM + locality if available)
X, y, ids, GRM_df, locality, code_to_label, y_eval = load_data(
    paths, target_column=target_col, standardize_features=base.get('standardize_features', False),
    return_locality=True, min_count=20, return_eval=True, eval_target_column=eval_col,
 )
if y_eval is None:
    y_eval = y.copy()

# Graph config baseline (can be overridden interactively)
gspace: Dict[str, Any] = CFG.get('search_space', {}).get('graph', {})
gcfg = {
    'graph_on': True,
    'source': gspace.get('source_default', 'grm'),
    'knn_k': int(gspace.get('knn_k_default', 10)),
    'weighted_edges': bool(gspace.get('weighted_edges_default', True)),
    'symmetrize_mode': gspace.get('symmetrize_mode_default', 'union'),
    'self_loops': True,
    'laplacian_smoothing': False,
}

# Build adjacency on all nodes
A = build_adjacency(X, GRM_df, gcfg, node_idx=None)
edge_index, edge_weight, _ = to_sparse(A, device=torch.device('cpu'))

n = X.shape[0]
print('Adjacency:', A.shape, 'nnz=', A.nnz)

Building global adjacency from source: grm
Distance matrix:
[[0.62167012 1.12526969 1.62491857 ... 1.54017174 1.61855312 1.62949845]
 [1.12526969 0.61374503 1.60143029 ... 1.60408675 1.62980842 1.62997147]
 [1.62491857 1.60143029 0.6336324  ... 1.63463229 1.62809941 1.61816341]
 ...
 [1.54017174 1.60408675 1.63463229 ... 0.63203757 1.62349742 1.64553773]
 [1.61855312 1.62980842 1.62809941 ... 1.62349742 0.60904647 1.63950873]
 [1.62949845 1.62997147 1.61816341 ... 1.64553773 1.63950873 0.62238316]]


KeyboardInterrupt: 

In [ ]:
GRM_df

,8L19505,8L19506,8L19540,8L19549,8L19556,8L19559,8L19568,8L19569,8L19578,8L19595,...,8981929,8981944,8981945,8981955,8981956,8981962,8981965,8981971,8981972,8L49911
8L19505,1.009037,0.505437,0.005789,0.016404,0.024300,-0.003731,0.010319,-0.007076,0.001272,0.020936,...,0.058304,-0.000124,0.003813,0.006854,0.001228,-0.005232,-0.003774,0.090535,0.012154,0.001209
8L19506,0.505437,1.016962,0.029277,0.009560,0.019356,-0.002775,0.001669,0.002900,0.000933,0.023489,...,0.020414,0.003324,0.003408,0.029825,-0.010731,-0.020577,-0.023779,0.026620,0.000899,0.000736
8L19540,0.005789,0.029277,0.997075,0.003206,-0.013655,0.003026,0.032947,0.028973,-0.014577,-0.009090,...,0.074652,-0.006602,-0.017213,0.494489,-0.015652,-0.012710,-0.002731,-0.003925,0.002608,0.012544
8L19549,0.016404,0.009560,0.003206,1.020620,0.009898,0.005117,0.025466,-0.011768,0.011960,0.052632,...,0.024469,-0.010527,-0.002416,-0.016684,-0.006214,-0.001185,-0.004926,0.004132,0.044847,-0.008049
8L19556,0.024300,0.019356,-0.013655,0.009898,0.998242,0.020314,-0.001224,-0.012694,0.017325,-0.020716,...,0.008259,-0.002534,-0.002325,0.006756,-0.005960,0.005769,-0.011063,-0.002157,0.009938,0.004236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8981962,-0.005232,-0.020577,-0.012710,-0.001185,0.005769,0.001793,0.014383,0.087584,0.050145,0.024818,...,-0.014690,0.017701,0.206575,-0.018125,0.281999,1.203145,0.657679,-0.008266,0.105940,-0.014916
8981965,-0.003774,-0.023779,-0.002731,-0.004926,-0.011063,0.006754,0.002204,0.106372,0.092120,0.028310,...,-0.010461,0.021201,0.267484,0.004365,0.423612,0.657679,1.175029,-0.003767,0.083535,-0.006325
8981971,0.090535,0.026620,-0.003925,0.004132,-0.002157,0.007203,0.019337,0.029563,0.003768,0.006225,...,0.035346,0.000064,-0.006160,-0.006207,0.007410,-0.008266,-0.003767,0.998670,0.007210,-0.014831
8981972,0.012154,0.000899,0.002608,0.044847,0.009938,-0.013065,-0.003451,0.067659,0.046565,0.038242,...,0.159708,0.018696,0.069247,-0.006797,0.116962,0.105940,0.083535,0.007210,1.021661,-0.008802


In [ ]:
B = A.toarray()
print(B.shape)
B[B==0] = np.nan
B[B==1] = np.nan
B[B<0.6] = np.nan
#delete rows with all nan
B = B[~np.isnan(B).all(axis=1)]
print(B.shape)

(5650, 5650)
(1056, 5650)


In [ ]:
# Homophily: corr(y_i, mean(y_neighbors)) for varying k (Euclidean, GRM-kNN, and GRM-threshold)
from sklearn.neighbors import NearestNeighbors

def homophily_corr(y: np.ndarray, X: np.ndarray, GRM_df, ks=(3,5,7,10,20), metric='euclidean', grm_thresholds=(0.2,0.3,0.4,0.5,0.6)) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - df_k: columns [k, corr_euclidain, corr_grm] using kNN (self excluded)
      - df_thr: columns [cutoff, correlation] using GRM threshold graph (self excluded)
    """
    y = y.astype(np.float64).ravel()
    rows_k = []
    # Prepare GRM-based similarity and distance if available
    G_norm = None
    dist_grm = None
    if GRM_df is not None:
        G = GRM_df.to_numpy().astype(np.float64)
        diag = np.clip(np.diag(G), 1e-12, None)
        D = np.sqrt(np.outer(diag, diag))
        G_norm = np.clip(G / D, -1.0, 1.0)
        dist_grm = 1.0 - G_norm
    # GRM-threshold homophily (independent of k)
    thr_rows = []
    if G_norm is not None:
        # ensure self is excluded from neighbor sets
        G_no_self = G_norm.copy()
        np.fill_diagonal(G_no_self, -np.inf)
        for t in grm_thresholds:
            mask = (G_no_self >= float(t))  # boolean adjacency (no self)
            deg = mask.sum(axis=1)
            with np.errstate(invalid='ignore', divide='ignore'):
                y_sum = mask @ y
                y_mean = np.where(deg > 0, y_sum / deg, np.nan)
            valid = ~np.isnan(y_mean)
            if valid.sum() > 1 and np.std(y_mean[valid]) > 0 and np.std(y[valid]) > 0:
                r_t = float(np.corrcoef(y[valid], y_mean[valid])[0,1])
            else:
                r_t = float('nan')
            thr_rows.append({'cutoff': float(t), 'correlation': r_t})
    else:
        thr_rows = [{'cutoff': float(t), 'correlation': float('nan')} for t in grm_thresholds]
    # k-dependent homophily (Euclidean + GRM-kNN)
    for k in ks:
        if k <= 0:
            continue
        # Euclidean kNN on features; use k+1 then drop self (column 0)
        nbrs_euc = NearestNeighbors(n_neighbors=k+1, metric=metric).fit(X)
        _, ind_euc = nbrs_euc.kneighbors(X)
        idx_euc = ind_euc[:, 1:]  # self excluded
        y_neigh_mean_euc = y[idx_euc].mean(axis=1)
        r_euc = np.corrcoef(y, y_neigh_mean_euc)[0,1] if np.std(y_neigh_mean_euc)>0 and np.std(y)>0 else 0.0
        # GRM kNN using precomputed distance (1 - normalized GRM); drop self as above
        if dist_grm is not None:
            nbrs_grm = NearestNeighbors(n_neighbors=k+1, metric='precomputed').fit(dist_grm)
            _, ind_grm = nbrs_grm.kneighbors(dist_grm)
            idx_grm = ind_grm[:, 1:]  # self excluded (dist 0 on diagonal)
            y_neigh_mean_grm = y[idx_grm].mean(axis=1)
            r_grm = np.corrcoef(y, y_neigh_mean_grm)[0,1] if np.std(y_neigh_mean_grm)>0 and np.std(y)>0 else 0.0
        else:
            r_grm = float('nan')
        rows_k.append({'k': int(k), 'corr_euclidain': float(r_euc), 'corr_grm': float(r_grm)})
    df_k = pd.DataFrame(rows_k)
    df_thr = pd.DataFrame(thr_rows)
    return df_k, df_thr

hom_df, hom_thr_df = homophily_corr(y, X, GRM_df, ks=(1,3,5,7,10,15,30))
hom_df, hom_thr_df

(    k  corr_euclidain  corr_grm
 0   1        0.251486  0.242760
 1   3        0.321252  0.320201
 2   5        0.346198  0.343836
 3   7        0.348005  0.346553
 4  10        0.344364  0.340079
 5  15        0.330092  0.331834
 6  30        0.295998  0.305238,
    cutoff  correlation
 0     0.2     0.302422
 1     0.3     0.320094
 2     0.4     0.331348
 3     0.5     0.315564
 4     0.6     0.395335)

In [ ]:
# Moran's I for adjusted phenotype on the built graph
def morans_I(y: np.ndarray, A_csr: sp.csr_matrix) -> float:
    y = y.astype(np.float64).ravel()
    y_c = y - y.mean()
    # Weight matrix W = A without diagonal
    A = A_csr.copy().tocsr()
    A.setdiag(0)
    A.eliminate_zeros()
    W = A
    w_sum = W.sum()
    if w_sum == 0:
        return 0.0
    num = y_c @ (W @ y_c)
    den = (y_c ** 2).sum()
    n = y_c.size
    return float((n / w_sum) * (num / den)) if den > 0 else 0.0

I = morans_I(y, A)
print('Moran\'s I:', I)

Moran's I: 0.2017385544717356


In [ ]:
# Neighbor-mean baseline vs simple MLP
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor

def neighbor_mean_predict(X: np.ndarray, y: np.ndarray, train_idx: np.ndarray, test_idx: np.ndarray, k: int = 10, metric: str = 'euclidean') -> np.ndarray:
    nbrs = NearestNeighbors(n_neighbors=k, metric=metric).fit(X[train_idx])
    d, ind = nbrs.kneighbors(X[test_idx])
    # weighted average by inverse distance; fall back to uniform if zeros
    w = 1.0 / (d + 1e-8)
    w = w / w.sum(axis=1, keepdims=True)
    return (w * y[train_idx][ind]).sum(axis=1)

def evaluate_baselines(X: np.ndarray, y: np.ndarray, k_list=(5,10,20), n_splits=5, seed=42) -> pd.DataFrame:
    rows = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k in k_list:
        r_neigh, r_mlp = [], []
        for tr, te in kf.split(X):
            y_pred_neigh = neighbor_mean_predict(X, y, tr, te, k=k)
            r_neigh.append(_pearson_corr(y[te], y_pred_neigh))
            mlp = MLPRegressor(hidden_layer_sizes=(128,64), activation='relu', solver='adam', learning_rate_init=1e-3, max_iter=500, random_state=seed)
            mlp.fit(X[tr], y[tr].ravel())
            r_mlp.append(_pearson_corr(y[te], mlp.predict(X[te]).ravel()))
        rows.append({'k': int(k), 'neighbor_mean_corr_mean': float(np.mean(r_neigh)), 'neighbor_mean_corr_std': float(np.std(r_neigh)),
                     'mlp_corr_mean': float(np.mean(r_mlp)), 'mlp_corr_std': float(np.std(r_mlp))})
    return pd.DataFrame(rows)

baseline_df = evaluate_baselines(X, y, k_list=(5,10,20), n_splits=5)
baseline_df

c:\Users\Simen\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:697: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
